In [1]:
import os


In [2]:
%pwd

'/mnt/d/resume_projects/flight_fare_prediction/research'

In [3]:
os.chdir("../")
%pwd

'/mnt/d/resume_projects/flight_fare_prediction'

In [4]:
import pandas as pd
import numpy as np


In [5]:
df = pd.read_csv("artifacts/data_ingestion/feature_store/flight_fare.csv")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32046 entries, 0 to 32045
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          32046 non-null  object
 1   Date_of_Journey  32046 non-null  object
 2   Source           32046 non-null  object
 3   Destination      32046 non-null  object
 4   Route            32043 non-null  object
 5   Dep_Time         32046 non-null  object
 6   Arrival_Time     32046 non-null  object
 7   Duration         32046 non-null  object
 8   Total_Stops      32043 non-null  object
 9   Additional_Info  32046 non-null  object
 10  Price            32046 non-null  int64 
dtypes: int64(1), object(10)
memory usage: 2.7+ MB


In [7]:
from dataclasses import dataclass
from pathlib import Path
from src.flight_price_prediction.constants import *

In [8]:
@dataclass
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: Path
    validated_dir: Path
    invalid_dir: Path
    drift_report_dir: Path
    drift_report_file_name: Path
    valid_train_file_name: Path
    valid_test_file_name: Path
    all_schema: dict

In [9]:
from src.flight_price_prediction.exception.exception import CustomException
from src.flight_price_prediction.logging.logger import logging
from src.flight_price_prediction.constants import *
from src.flight_price_prediction.entity.config_entity import DataValidationConfig
from src.flight_price_prediction.entity.artifact_entity import DataIngestionArtifact
import sys
import os
from src.flight_price_prediction.utils.common import *

In [10]:
# configuration manager
from src.flight_price_prediction.constants import *
from src.flight_price_prediction.utils.common import read_yaml, create_directories
from src.flight_price_prediction.entity.config_entity import DataIngestionConfig,DataValidationConfig
from pathlib import Path

class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        params = self.params.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            feature_store_file_name=Path(config.feature_store_file_name),
            training_file_name=Path(config.training_file_name),
            testing_file_name=Path(config.testing_file_name),
            train_test_split_ratio = params.train_test_split_ratio,
            collection_name=config.collection_name,
            database_name=config.database_name
        )

        return data_ingestion_config
    
    def get_data_validation_config(self) -> DataValidationConfig:
         config = self.config.data_validation
         schema = self.schema.COLUMNS

         data_validation_dir = Path(config.root_dir)
         validated_dir = Path(config.validated_dir)
         invalid_dir = Path(config.invalid_dir)
         drift_report_dir = Path(config.drift_report_dir)
         valid_train_file_name = Path(config.valid_train_file_name)
         valid_test_file_name = Path(config.valid_test_file_name)
         drift_report_file_name = Path(config.drift_report_file_name)
         Status_File = Path(config.Status_File)


         create_directories([data_validation_dir,validated_dir,invalid_dir,
                             drift_report_dir])
         data_validation_config = DataValidationConfig(
             root_dir = data_validation_dir,
             Status_File = Status_File,
             validated_dir = validated_dir,
             invalid_dir = invalid_dir,
             drift_report_dir = drift_report_dir,
             valid_train_file_name = valid_train_file_name,
             valid_test_file_name = valid_test_file_name,
             drift_report_file_name = drift_report_file_name,
             all_schema = schema,
           )
         return data_validation_config
         
         

In [ ]:
#data validation component
import sys
from pathlib import Path
from src.flight_price_prediction.entity.config_entity import DataValidationConfig
from src.flight_price_prediction.entity.artifact_entity import DataIngestionArtifact,DataValidationArtifact
from src.flight_price_prediction.exception.exception import CustomException
from src.flight_price_prediction.logging.logger import logging
from src.flight_price_prediction.utils.common import *
from scipy.stats import chi2_contingency
import pandas as pd

class DataValidation:
    def __init__(self, data_ingestion_artifact:DataIngestionArtifact,
                 data_validation_config:DataValidationConfig):
        try:
            self.data_ingestion_artifact = data_ingestion_artifact
            self.data_validation_config = data_validation_config
            self.schema = self.data_validation_config.all_schema
        except Exception as e:
            raise CustomException(e, sys)
    
    @staticmethod  
    def read_data(file_path) -> pd.DataFrame:
        try:
            return pd.read_csv(file_path)
        except Exception as e:
            raise CustomException(e, sys)
    
    def impute_null_values(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        """Impute null values using mode imputation"""
        try:
            if dataframe.isnull().sum().sum() == 0:
                logging.info("No null values found in dataframe")
                return dataframe
            
            logging.info("Null values detected. Imputing using mode")
            
            # Create a copy to avoid modifying original
            df_imputed = dataframe.copy()
            
            # Fill all columns with mode
            for col in df_imputed.columns:
                if df_imputed[col].isnull().sum() > 0:
                    mode_value = df_imputed[col].mode()[0] if len(df_imputed[col].mode()) > 0 else 'Unknown'
                    df_imputed[col] = df_imputed[col].fillna(mode_value)
                    logging.info(f"Filled null values in {col} with mode: {mode_value}")
            
            logging.info("All null values have been imputed successfully")
            return df_imputed
        except Exception as e:
            raise CustomException(e, sys)
    
    def validate_columns(self, dataframe: pd.DataFrame) -> bool:
        try:
            
            expected_columns = list(self.schema.keys())
            target_info = self.schema.get('TARGET_COLUMN', {})
            target_column = target_info.get('name', 'Price')
            status = True
            schema_dtypes = self.schema.get('COLUMNS_TYPE', {})
            if target_column not in expected_columns:
                expected_columns.append(target_column)
            #check no. of columns
            if len(dataframe.columns) != len(expected_columns):
                logging.error(f"column count mismatch. Expected: {len(expected_columns)}")
                return False
            
            #check for same columns present of different columns
            if set(dataframe.columns) != set(expected_columns):
                missing_cols = set(expected_columns) -set(dataframe.columns)
                extra_cols = set(dataframe.columns) - set(expected_columns)
                logging.error(f"column name mismatch. Missing : {missing_cols}, Extra: {extra_cols}")
                return False
            #check for datatypes
            for col, expected_dtype in schema_dtypes.items():
                if col in dataframe.columns:
                    actual_dtype = str(dataframe[col].dtype)
                    if actual_dtype != expected_dtype:
                        logging.error(f"Data type mismatch for column: {col}. "
                                      f"Expected: {expected_dtype}, Found: {actual_dtype}")
                        return False
            logging.info("schema validation successful: all columns match")
            return True
    
        except Exception as e:
            raise CustomException(e, sys)
        
    def detect_dataset_drift(self, base_df, current_df, threshold=0.05) -> bool:
        try:
            status = True
            report = {}
            
            # Select only categorical columns for Chi-Square
            cat_cols = base_df.select_dtypes(include=['object']).columns
            
            for column in cat_cols:
                # 1. Get frequencies for both datasets
                base_counts = base_df[column].value_counts()
                current_counts = current_df[column].value_counts()
                
                # 2. Align the data (ensure both datasets have the same categories)
                df_counts = pd.DataFrame({
                    'base': base_counts,
                    'current': current_counts
                }).fillna(0) # Fill missing categories with 0
                
                # 3. Perform Chi-Square test on the aligned table
                chi2, p_value, dof, expected_freq = chi2_contingency(df_counts)
                
                is_found = p_value < threshold
                if is_found:
                    status = False
                
                report[column] = {
                    "p_value": float(p_value),
                    "drift_status": bool(is_found)
                }
                logging.info(f"Drift check for {column}: p_value={p_value:.4f}, drift={is_found}")
            
            # Save report
            write_yaml_file(file_path=str(self.data_validation_config.drift_report_file_name), content=report)
            return status
            
        except Exception as e:
            raise CustomException(e, sys)
        
    def initiate_data_validation(self) -> DataValidationArtifact:
        try:
            train_file_name = self.data_ingestion_artifact.training_file_name
            test_file_name = self.data_ingestion_artifact.testing_file_name

            # Read the data from train and test
            train_dataframe = DataValidation.read_data(train_file_name)
            test_dataframe = DataValidation.read_data(test_file_name)

            # Validate number of columns
            status = self.validate_columns(dataframe=train_dataframe)
            if not status:
                error_message = f"train dataframe does not contain all columns.\n"
            status = self.validate_columns(dataframe=test_dataframe)
            if not status:
                error_message = f"test dataframe does not contain all the columns.\n"
            
            # Impute null values using mode
            train_dataframe = self.impute_null_values(train_dataframe)
            test_dataframe = self.impute_null_values(test_dataframe)
            
            # Check dataset drift
            status = self.detect_dataset_drift(base_df=train_dataframe, current_df=test_dataframe)
            
            # Create and save the data
            save_data(train_dataframe, self.data_validation_config.valid_train_file_name)
            save_data(test_dataframe, self.data_validation_config.valid_test_file_name)

            # Write validation status to STATUS_FILE
            status_report = {
                "validation_status": status,
                "message": "Data validation passed" if status else "Data validation failed - drift detected"
            }
            write_yaml_file(file_path=str(self.data_validation_config.Status_File), content=status_report)
            logging.info(f"Validation status saved to {self.data_validation_config.Status_File}")

            data_validation_artifact = DataValidationArtifact(
                validation_status=Path(self.data_validation_config.Status_File),
                valid_train_file_name=Path(self.data_validation_config.valid_train_file_name),
                valid_test_file_name=Path(self.data_validation_config.valid_test_file_name),
                invalid_train_file_name=None,
                invalid_test_file_name=None,
                drift_report_file_name=Path(self.data_validation_config.drift_report_file_name))
            return data_validation_artifact
        except Exception as e:
            raise CustomException(e, sys)


In [12]:
#data validation pipeline
from src.flight_price_prediction.config.configuration import ConfigurationManager
from src.flight_price_prediction.entity.artifact_entity import DataIngestionArtifact
from src.flight_price_prediction.pipeline.data_ingestion_pipeline import DataIngestionTrainingPipeline

from src.flight_price_prediction.logging.logger import logging
from src.flight_price_prediction.exception.exception import CustomException


In [13]:
STAGE = "Data Validation Stage"

class DataValidationTrainingPipeline:
    def __init__(self, config: ConfigurationManager,data_ingestion_artifact: DataIngestionArtifact):
        try:
            self.config = config
            self.data_ingestion_artifact = data_ingestion_artifact
        except Exception as e:
            raise CustomException(e, sys)
        

    def initiate_data_validation(self):
        try:
            data_validation_config = self.config.get_data_validation_config()
            data_validation = DataValidation(
                data_ingestion_artifact = self.data_ingestion_artifact,
                data_validation_config = data_validation_config
            )
            return data_validation.initiate_data_validation()
        except Exception as e:
            raise CustomException(e, sys)
        
if __name__=="__main__":
    try:
        logging.info(f">>>> stage {STAGE} started <<<<")
        config = ConfigurationManager()
        ingestion_pipeline = DataIngestionTrainingPipeline(config=config)
        data_ingestion_artifact = ingestion_pipeline.initiate_data_ingestion()
        obj = DataValidationTrainingPipeline(config =config, data_ingestion_artifact= data_ingestion_artifact)
        obj.initiate_data_validation()
        logging.info(f">>>> stage {STAGE} completed <<<<")
    except Exception as e:
        raise CustomException(e, sys)

[2026-06-04 17:51:23,740: INFO: 1876129590: >>>> stage Data Validation Stage started <<<<]
[2026-06-04 17:51:23,747: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/config/config.yaml loaded succesfully ]
[2026-06-04 17:51:23,755: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/params/params.yaml loaded succesfully ]
[2026-06-04 17:51:23,769: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/schema/schema.yaml loaded succesfully ]
[2026-06-04 17:51:23,774: INFO: common: created directory at: artifacts]
[2026-06-04 17:51:23,783: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-04 17:51:39,340: INFO: common: Data saved to: artifacts/data_ingestion/feature_store/flight_fare.csv]
[2026-06-04 17:51:39,342: INFO: data_ingestion: saved raw data into feature store file path: artifacts/data_ingestion/feature_store/flight_fare.csv]
[2026-06-04 17:51:39,345: INFO: data_ingestion: splitting data into train_tes